In [0]:
from pyspark.sql.functions import lit , row_number
from pyspark.sql.window import Window
data = [(1,),(55,)]
schema = 'age int'

df = spark.createDataFrame(data , schema)
age = df.collect()
num = age[0]["age"]



df.withColumn("age" , row_number().over(Window.orderBy(lit(1))) + lit(num)).show()



In [0]:
import os

os.environ.get("demo")



In [0]:
%sql
select * from dev.spark_db.order_msg;

In [0]:
%sql
select * from dev.spark_db.order;

In [0]:
%sql
select * from dev.spark_db.order_msg;

In [0]:
%sql
select * from dev.spark_db.asset_rate;

In [0]:
%sql

select * from dev.spark_db.order;

In [0]:
%sql
select * from dev.spark_db.order_error;

In [0]:
%sql
select * from dev.spark_db.order_stg;

In [0]:
%sql
select * from dev.spark_db.transaction;

In [0]:
%sql
select * from dev.spark_db.quantity where account_number = '06932-99420' and asset_external_id = 'AAPL';

In [0]:
%sql
select * from dev.spark_db.quantity where account_number = '97940-69402' and asset_external_id = 'ICICIBANK';

In [0]:
%sql
select * from dev.spark_db.eod_market_value;

In [0]:
from pyspark.sql.window import Window



In [0]:
%sql
select * from (
select account_number,
       asset_external_id,
       quantity,
       dense_rank() over(partition by account_number,asset_external_id order by quantity desc) as rnk
 from dev.spark_db.order
)
where rnk =2; 

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank,col

(
spark.read.table("dev.spark_db.order").withColumn(
    "rnk" , dense_rank().over(Window.partitionBy("account_number" , "asset_external_id")
                              .orderBy(col("quantity").desc())
                              )
).where(col("rnk")==2)
.select("account_number",
         "asset_external_id",
         "quantity",
         "rnk")
).display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank,col,lag,first

(
spark.read.table("dev.spark_db.order").withColumn(
    "rnk" , first("quantity").over(Window.partitionBy("account_number" , "asset_external_id")
                              .orderBy(col("quantity").desc())
                              )
).select("account_number",
         "asset_external_id",
         "quantity",
         "rnk")
).display()

In [0]:
from pyspark.sql.functions import col,explode
from pyspark.sql.types import StructType,StructField,IntegerType,ArrayType,StringType,MapType
data = [(1,['Luffy' , 'Zero'],{"Role" :"Captain","Bounty":"567"}),
        (2,['Nami','Ussop'],{"Role" : "Gods" , "Bounty" : "789"})
        ]
schema = StructType(
    [
        StructField("Rank" , IntegerType(), False),
        StructField("Pirates" , ArrayType(StringType()) , False),
        StructField(
                    "Detail" , MapType(
                                        StringType(),StringType()
                                        )
                    )
    ]
)

df = spark.createDataFrame(data,schema)
df = df.withColumn("Single0" , col("Pirates")[0])
df = df.withColumn("Single1" , col("Pirates")[1])
df = df.withColumn("Detail1" , col("Detail")["Role"])
df.display()

In [0]:
%sql
CREATE CATALOG test;

In [0]:
%sql
drop volume test.dev.word

In [0]:
df =  spark.read.format("csv").option("header", "true").load("/Volumes/test/dev/sales/sales1.csv")
df.display()

In [0]:
%sql
select * from read_files("/Volumes/test/dev/sales/sales1.csv",
                            format =>'csv',
                            header=>True);
                            

In [0]:
#schemaEvolutionMode
    addNewColumns
    rescue
    failAllColumns
    None

In [0]:
df = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format" , "csv")\
                .option("cloudFiles.schemaLocation" , "/Volumes/test/dev/sales/schema/sale/")\
                .option("cloudFiles.schemaEvolutionMode" , "rescue")\
                .load("/Volumes/test/dev/sales/soucre/sales1.csv")
                
df.writeStream.format("delta")\
              .option("checkpointLocation" , '/Volumes/test/dev/sales/checkpoint/sale/')\
              .trigger(availableNow = True)\
              .start("/Volumes/test/dev/sales/destination/sale/")

In [0]:
df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option(
             "cloudFiles.schemaLocation",
             "/Volumes/test/dev/sales/schema/sale/"
         )
         .option("cloudFiles.schemaEvolutionMode", "rescue")
         .load("/Volumes/test/dev/sales/soucre/")
)



In [0]:
query = (
    df.writeStream
      .format("delta")
      .option(
          "checkpointLocation",
          "/Volumes/test/dev/sales/checkpoint/sale/"
      )
      .trigger(availableNow=True)
      .start("/Volumes/test/dev/sales/destination/sale/")
)

query.awaitTermination()

In [0]:
%sql
select * from delta.`/Volumes/test/dev/sales/destination/sale/`;

In [0]:
df = spark.read.format("csv").option("header", "true").load('/Volumes/test/dev/sales/soucre/sales1.csv')

In [0]:
from pyspark.sql.functions import col,cast,round,expr,dense_rank,lit
from pyspark.sql.types import DecimalType
from pyspark.sql.window import Window
d_df = (
     df.withColumn("total_quantity" , col("Quantity").cast( DecimalType(10,2)) * col("Price").cast(DecimalType(10,2)))
 )

d_df.groupBy(to_date("OrderDate")).agg(sum(col("total_quantity")).alias("monthly_revenue")).display()

In [0]:
from pyspark.sql.functions import to_date,sum

(
    df.groupBy(
                col("Item")
              ).
                    agg(
                        sum(col("Quantity")).alias("total_quantity")
                        ).display()
)

In [0]:
df.select(to_date(col("OrderDate"))).display()

In [0]:
%sql
drop catalog demo_conn_catalog;

In [0]:
%sql
select * from demo_plsql_conn_catalog.public.account;

In [0]:
dbutils.jobs.taskValues.set(key = 'lookup_output', value = 'Toys')

{'lookup_output' : 'Toys'}